In [1]:
import redis
import time

def test_redis():
    # 1. 建立连接
    r = redis.Redis(
        host='localhost',
        port=6379,
        db=0,
        decode_responses=True  # 自动将字节解码为字符串，推荐开启
    )

    print("=== 连接测试 ===")
    try:
        pong = r.ping()
        print(f"PING 响应: {pong}")
    except redis.ConnectionError as e:
        print(f"连接失败: {e}")
        return

    # 2. String 操作
    print("\n=== String 操作 ===")
    r.set("user:name", "Alice")
    name = r.get("user:name")
    print(f"GET user:name -> {name}")

    # 带过期时间的 SET (EX=秒)
    r.set("session:token", "abc123", ex=10)
    ttl = r.ttl("session:token")
    print(f"session:token TTL -> {ttl}s")

    # INCR / DECR 计数器
    r.set("page:views", 0)
    r.incr("page:views", amount=5)
    print(f"page:views -> {r.get('page:views')}")

    # 3. Hash 操作
    print("\n=== Hash 操作 ===")
    r.hset("user:1001", mapping={
        "name": "Bob",
        "age": "28",
        "email": "bob@example.com"
    })
    user = r.hgetall("user:1001")
    print(f"HGETALL user:1001 -> {user}")
    print(f"HGET age -> {r.hget('user:1001', 'age')}")

    # 4. List 操作
    print("\n=== List 操作 ===")
    r.delete("task:queue")  # 清理旧数据
    r.rpush("task:queue", "send_email", "gen_report", "sync_data")
    length = r.llen("task:queue")
    first_task = r.lpop("task:queue")
    print(f"LLEN -> {length}, LPOP -> {first_task}")
    print(f"LRANGE 剩余 -> {r.lrange('task:queue', 0, -1)}")

    # 5. Set 操作
    print("\n=== Set 操作 ===")
    r.sadd("tags:article:1", "python", "redis", "backend")
    r.sadd("tags:article:2", "redis", "database", "nosql")
    common = r.sinter("tags:article:1", "tags:article:2")
    all_tags = r.sunion("tags:article:1", "tags:article:2")
    print(f"SINTER 交集 -> {common}")
    print(f"SUNION 并集 -> {all_tags}")

    # 6. Pipeline 批量操作（减少网络往返）
    print("\n=== Pipeline 批量写入 ===")
    pipe = r.pipeline(transaction=True)
    for i in range(10):
        pipe.set(f"batch:key:{i}", f"value_{i}")
    results = pipe.execute()
    print(f"Pipeline 执行结果数: {len(results)}, 全部成功: {all(results)}")

    # 7. Key 模式搜索与清理
    print("\n=== 清理测试数据 ===")
    keys = r.keys("user:*") + r.keys("session:*") + \
           r.keys("page:*") + r.keys("task:*") + \
           r.keys("tags:*") + r.keys("batch:*")
    if keys:
        deleted = r.delete(*keys)
        print(f"已删除 {deleted} 个测试 key")
    else:
        print("无测试 key 需要清理")

    print("\n✅ Redis 基础功能测试完成!")


if __name__ == "__main__":
    test_redis()


=== 连接测试 ===
PING 响应: True

=== String 操作 ===
GET user:name -> Alice
session:token TTL -> 10s
page:views -> 5

=== Hash 操作 ===
HGETALL user:1001 -> {'name': 'Bob', 'age': '28', 'email': 'bob@example.com'}
HGET age -> 28

=== List 操作 ===
LLEN -> 3, LPOP -> send_email
LRANGE 剩余 -> ['gen_report', 'sync_data']

=== Set 操作 ===
SINTER 交集 -> {'redis'}
SUNION 并集 -> {'python', 'redis', 'nosql', 'backend', 'database'}

=== Pipeline 批量写入 ===
Pipeline 执行结果数: 10, 全部成功: True

=== 清理测试数据 ===
已删除 17 个测试 key

✅ Redis 基础功能测试完成!


In [3]:
from confluent_kafka import Producer, Consumer, KafkaError
import json
import time
import uuid

BOOTSTRAP_SERVERS = "localhost:9092"
TEST_TOPIC = "test_python_topic"


# ==================== 生产者 ====================
def create_producer():
    conf = {
        'bootstrap.servers': BOOTSTRAP_SERVERS,
        'client.id': 'py-test-producer',
        # acks=all: 所有 ISR 副本确认后才返回成功，保证不丢消息
        'acks': 'all',
        # 消息压缩
        'compression.type': 'snappy',
    }
    return Producer(conf)


def delivery_callback(err, msg):
    """异步发送回调：每条消息投递后触发"""
    if err:
        print(f"❌ 投递失败: {err}")
    else:
        print(f"✅ 投递成功: topic={msg.topic()}, "
              f"partition={msg.partition()}, offset={msg.offset()}")


def test_producer():
    print("=== 生产者测试 ===")
    producer = create_producer()

    # 1. 简单字符串消息
    producer.produce(
        TEST_TOPIC,
        key="user-001",
        value="hello kafka",
        on_delivery=delivery_callback
    )

    # 2. JSON 序列化消息 + 自定义 Headers
    payload = {"event": "order_created", "order_id": str(uuid.uuid4()), "amount": 99.5}
    producer.produce(
        TEST_TOPIC,
        key=payload["order_id"],
        value=json.dumps(payload).encode("utf-8"),
        headers=[("source", b"python-test"), ("version", b"v1")],
        on_delivery=delivery_callback
    )

    # 3. 指定分区发送
    producer.produce(
        TEST_TOPIC,
        partition=0,
        key="fixed-partition",
        value="this goes to partition 0",
        on_delivery=delivery_callback
    )

    # flush 阻塞等待所有消息投递完成（超时10秒）
    remaining = producer.flush(timeout=10)
    print(f"未投递消息数: {remaining}")
    print()


# ==================== 消费者 ====================
def test_consumer(max_messages=3, timeout=15):
    print("=== 消费者测试 ===")
    conf = {
        'bootstrap.servers': BOOTSTRAP_SERVERS,
        'group.id': 'py-test-consumer-group',
        # earliest: 从最早位点开始消费；latest: 只消费新消息
        'auto.offset.reset': 'earliest',
        'enable.auto.commit': False,  # 手动提交，演示 commit 流程
        'client.id': 'py-test-consumer',
    }
    consumer = Consumer(conf)
    consumer.subscribe([TEST_TOPIC])

    consumed = 0
    start_time = time.time()
    print(f"等待最多 {max_messages} 条消息 (超时 {timeout}s)...")

    try:
        while consumed < max_messages:
            if time.time() - start_time > timeout:
                print("⏰ 消费超时，退出")
                break

            msg = consumer.poll(timeout=1.0)
            if msg is None:
                continue
            if msg.error():
                if msg.error().code() == KafkaError._PARTITION_EOF:
                    print(f"[EOF] {msg.topic()}:{msg.partition()} 已到末尾")
                else:
                    print(f"❌ 消费错误: {msg.error()}")
                continue

            # 解析消息
            key = msg.key().decode("utf-8") if msg.key() else None
            value = msg.value().decode("utf-8") if msg.value() else None
            headers = dict(msg.headers()) if msg.headers() else {}

            print(f"📨 收到消息: partition={msg.partition()}, offset={msg.offset()}, "
                  f"key={key}, value={value[:80]}..., headers={headers}")

            # 手动提交当前位点
            consumer.commit(message=msg, asynchronous=False)
            consumed += 1

    except KeyboardInterrupt:
        print("\n用户中断")
    finally:
        consumer.close()
        print(f"消费者已关闭，共消费 {consumed} 条消息")


# ==================== 主入口 ====================
if __name__ == "__main__":
    test_producer()
    test_consumer()

=== 生产者测试 ===
✅ 投递成功: topic=test_python_topic, partition=0, offset=0
✅ 投递成功: topic=test_python_topic, partition=0, offset=1
✅ 投递成功: topic=test_python_topic, partition=0, offset=2
未投递消息数: 0

=== 消费者测试 ===
等待最多 3 条消息 (超时 15s)...
📨 收到消息: partition=0, offset=0, key=user-001, value=hello kafka..., headers={}
📨 收到消息: partition=0, offset=1, key=7ee6976d-39c2-4da2-85da-6568f135c940, value={"event": "order_created", "order_id": "7ee6976d-39c2-4da2-85da-6568f135c940", "..., headers={'source': b'python-test', 'version': b'v1'}
📨 收到消息: partition=0, offset=2, key=fixed-partition, value=this goes to partition 0..., headers={}
消费者已关闭，共消费 3 条消息
